# Task 4.1 — Effect of Thrust on Trajectory

**Classical Mechanics · Project I · EETAC, UPC**

Compares three thrust levels (T = 1000, 1500, 3000 N) with identical propellant mass.
Answers: maximum altitude, troposphere escape, and burnout time scaling.

In [ ]:
# ── Download simulation dependencies from GitHub ─────────────────────────────
import urllib.request, os
BASE = "https://raw.githubusercontent.com/gdlcjoel-lab/PROYECTOMECANICA/main/Definitivos/"
for f in ["rocket_simulation.py", "plot_utils.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)
        print(f"Downloaded {f}")

# Use non-interactive backend so plots render inline in Colab
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
print("Setup complete ✓")

In [ ]:
import numpy as np
import sys
from rocket_simulation import simulate_rocket, compute_metrics, print_metrics, G0, ISP, MP

THRUST_CASES = [1000.0, 1500.0, 3000.0]
LABELS       = ["T = 1000 N", "T = 1500 N (base)", "T = 3000 N"]
COLORS       = ["#5B9BD5", "#0D2B55", "#C0392B"]
PROP_MASS    = 90.0

print("=" * 60)
print("  TASK 4.1 — THRUST VARIATION")
print("=" * 60)

results, metrics_list = [], []
for T in THRUST_CASES:
    mdot   = T / (G0 * ISP)
    t_burn = PROP_MASS / mdot
    print(f"\n>>> T = {T:.0f} N  (mdot = {mdot:.4f} kg/s, t_burn = {t_burn:.2f} s)")
    t, r, v, drag, mass, accel = simulate_rocket(thrust=T, m_propellant=PROP_MASS)
    m_obj = compute_metrics(t, r, v)
    results.append((t, r, v, drag, mass, accel))
    metrics_list.append(m_obj)
    print_metrics(m_obj, f"T = {T:.0f} N")

print("\n1. HOW DOES THE MAXIMUM ALTITUDE CHANGE?")
for T, m in zip(THRUST_CASES, metrics_list):
    mdot = T / (G0 * ISP); tb = PROP_MASS / mdot
    print(f"   T={T:.0f}N  z_max={m['z_max']/1000:.3f} km  t_burn={tb:.2f} s")

print("\n2. DOES THE ROCKET ESCAPE THE TROPOSPHERE (>11 km)?")
for T, m in zip(THRUST_CASES, metrics_list):
    print(f"   T={T:.0f}N  z_max={m['z_max']/1000:.3f} km  ->  {'YES' if m['z_max']>11000 else 'NO'}")

print("\n3. HOW DOES BURNOUT TIME VARY?")
for T in THRUST_CASES:
    mdot = T / (G0 * ISP)
    print(f"   T={T:.0f}N  t_burn = {PROP_MASS/mdot:.2f} s")

## Plots
Altitude, speed, drag and mass vs time for all three thrust cases.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Task 4.1 — Effect of Thrust on Trajectory", fontsize=14, fontweight="bold", color="#0D2B55")
fig.patch.set_facecolor("white")
axes = axes.flatten()

ls = ["-", "--", ":"]
for i, (res, label, color, linestyle) in enumerate(zip(results, LABELS, COLORS, ls)):
    t, r, v, drag, mass, _ = res
    speed = np.linalg.norm(v, axis=1)
    axes[0].plot(t, r[:, 2]/1000, color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[1].plot(t, speed,         color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[2].plot(t, drag,          color=color, linestyle=linestyle, linewidth=2, label=label)
    axes[3].plot(t, mass,          color=color, linestyle=linestyle, linewidth=2, label=label)

titles  = ["Altitude vs Time", "Speed vs Time", "Aerodynamic Drag vs Time", "Mass vs Time"]
xlabels = ["Time [s]"] * 4
ylabels = ["Altitude [km]", "Speed [m/s]", "Drag Force [N]", "Mass [kg]"]
for ax, title, xl, yl in zip(axes, titles, xlabels, ylabels):
    ax.set_title(title, fontweight="bold", color="#0D2B55")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.4)

# Troposphere line on altitude panel
axes[0].axhline(11, color="gray", linestyle=":", linewidth=1, label="Troposphere (11 km)")
axes[0].legend(fontsize=8)

plt.tight_layout()
plt.savefig("task_4_1_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as task_4_1_plots.png")